In [1]:
import os
import glob
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import time

In [3]:
train_dir = "/content/drive/MyDrive/DeepFake Detection Project/Datasets/Batch_size_8/Train/"
validation_dir = "/content/drive/MyDrive/DeepFake Detection Project/Datasets/Batch_size_8/Validation/"

In [4]:
train_results = pd.read_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/Train_Metrics.xlsx")
validation_results = pd.read_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/Validation_Metrics.xlsx")

In [5]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Model
from keras.layers import (
    Input, Conv2D, MaxPooling2D, MaxPooling1D, Flatten, LSTM, ConvLSTM2D , Dense, TimeDistributed, Dropout,
    GlobalAveragePooling2D, Concatenate, AveragePooling2D, BatchNormalization, Average,
    Reshape, AveragePooling3D, MaxPooling3D, GlobalMaxPooling3D
)

In [6]:
def build_1PTCSL(input_shape=(10, 256, 256, 3), hidden_dim=64, num_classes=1):

    in1 = Input(shape=input_shape, name="Input_Layer")
    cnn = tf.keras.Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
    ], name="CNN_Module")


    x = TimeDistributed(cnn, name="Time_Distributed_CNN")(in1)
    x = ConvLSTM2D(hidden_dim, (3, 3), padding='same', return_sequences=True, name="ConvLSTM_Layer", activation='tanh')(x)
    x = GlobalMaxPooling3D()(x)
    x = Flatten(name="Flatten_Layer")(x)
    x = Dense(128, activation='relu', name="Dense_Layer_1")(x)
    x = Dropout(0.5, name="Dropout_Layer")(x)
    x = Dense(num_classes, activation='sigmoid', name="Output_Layer")(x)
    model = Model(inputs=in1, outputs=x, name="1PTCSL")
    model.summary()

    return model

In [ ]:
# Build & Compile Model Once
model = build_1PTCSL(input_shape=(10,256,256,3))
model.save_weights("/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/model.weights.h5")
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy','AUC','precision','recall']
)

In [ ]:
# Early‐Stopping Setup
best_val_loss = np.inf      # best validation loss seen so far
patience = 8                # how many epochs to wait after no improvement
wait = 0                    # counter for epochs with no improvement


for i in range(0,100):
    model = build_1PTCSL(input_shape=(10,256,256,3))
    model.load_weights(f"/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/model.weights.h5")
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy','AUC','precision','recall']
    )
    epoch_loss = []
    epoch_accuracy = []
    epoch_auc = []
    epoch_precision = []
    epoch_recall = []
    start_time = time.time()

    # Training Loop
    for j in range(0, 105):
        X0 = np.load(train_dir+f"Batch{j}X0.npy")
        print(f"Mounted Batch{j}X0")
        X1 = np.load(train_dir+f"Batch{j}X1.npy")
        print(f"Mounted Batch{j}X1")
        X2 = np.load(train_dir+f"Batch{j}X2.npy")
        print(f"Mounted Batch{j}X2")
        y = np.load(train_dir+f"Batch{j}y.npy")
        print(f"Mounted Batch{j}y")
        print(f"Epoch {i} of {100} ::: Training Batch {j+1} of {105}")
        history = model.fit(X0, y, epochs=1, verbose=0)

        # Collect training metrics
        epoch_loss.append(history.history['loss'][0])
        epoch_accuracy.append(history.history['accuracy'][0])
        epoch_auc.append(history.history['AUC'][0])
        epoch_precision.append(history.history['precision'][0])
        epoch_recall.append(history.history['recall'][0])
        print(f"Batch {i+1} Loss: {history.history['loss'][0]}; Accuracy: {history.history['accuracy'][0]}; AUC: {history.history['AUC'][0]}; Precision: {history.history['precision'][0]}; Recall: {history.history['recall'][0]} \n")

    # Validation Loop
    val_epoch_loss = []
    val_epoch_accuracy = []
    val_epoch_auc = []
    val_epoch_precision = []
    val_epoch_recall = []

    for j in range(0, 13):
        val_X0 = np.load(validation_dir+f"Batch{j}X0.npy")
        val_X1 = np.load(validation_dir+f"Batch{j}X1.npy")
        val_X2 = np.load(validation_dir+f"Batch{j}X2.npy")
        val_y = np.load(validation_dir+f"Batch{j}y.npy")
        print(f"Epoch {i} of {100} ::: Validation Batch {j+1} of 5")
        val_metrics = model.evaluate(val_X0, val_y, verbose=0)

        # Collect validation metrics
        val_epoch_loss.append(val_metrics[0])
        val_epoch_accuracy.append(val_metrics[1])
        val_epoch_auc.append(val_metrics[2])
        val_epoch_precision.append(val_metrics[3])
        val_epoch_recall.append(val_metrics[4])

    end_time = time.time()
    elapsed_time = end_time - start_time

    # Average the training metrics for the epoch
    loss_per_epoch = np.mean(epoch_loss)
    accuracy_per_epoch = np.mean(epoch_accuracy)
    auc_per_epoch = np.mean(epoch_auc)
    precision_per_epoch = np.mean(epoch_precision)
    recall_per_epoch = np.mean(epoch_recall)

    metrics = {
        'Loss': loss_per_epoch,
        'Accuracy': accuracy_per_epoch,
        'AUC': auc_per_epoch,
        'Precision': precision_per_epoch,
        'Recall': recall_per_epoch
    }
    train_results = pd.concat([train_results, pd.DataFrame([metrics])], ignore_index=True)
    train_results.to_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/Train_Metrics.xlsx", index=False)

    # Average the validation metrics for the epoch
    val_loss_per_epoch = np.mean(val_epoch_loss)
    val_accuracy_per_epoch = np.mean(val_epoch_accuracy)
    val_auc_per_epoch = np.mean(val_epoch_auc)
    val_precision_per_epoch = np.mean(val_epoch_precision)
    val_recall_per_epoch = np.mean(val_epoch_recall)

    metrics = {
        'Loss': val_loss_per_epoch,
        'Accuracy': val_accuracy_per_epoch,
        'AUC': val_auc_per_epoch,
        'Precision': val_precision_per_epoch,
        'Recall': val_recall_per_epoch
    }
    validation_results = pd.concat([validation_results, pd.DataFrame([metrics])], ignore_index=True)
    validation_results.to_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/Validation_Metrics.xlsx", index=False)

    # Print metrics for the epoch
    print(f"\nEpoch {i} completed in {elapsed_time:.2f} seconds")
    print(f"Training - Loss: {loss_per_epoch:.4f}, Accuracy: {accuracy_per_epoch:.4f}, "
          f"AUC: {auc_per_epoch:.4f}, Precision: {precision_per_epoch:.4f}, Recall: {recall_per_epoch:.4f}")
    print(f"Validation - Loss: {val_loss_per_epoch:.4f}, Accuracy: {val_accuracy_per_epoch:.4f}, "
          f"AUC: {val_auc_per_epoch:.4f}, Precision: {val_precision_per_epoch:.4f}, Recall: {val_recall_per_epoch:.4f}")

    # Early Stopping Check
    if val_loss_per_epoch < best_val_loss - 1e-3:
        best_val_loss = val_loss_per_epoch
        wait = 0
        # save best weights
        model.save_weights(
            "/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/best_model.weights.h5"
        )
    else:
        wait += 1
        if wait >= patience:
            print(f"\nEarly stopping at epoch {i+1}. "
                  f"Restoring best model from epoch {i+1-wait}.")
            model.load_weights(
                "/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/best_model.weights.h5"
            )
            break

    # Save checkpoint every epochs
    model.save_weights(f"/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/model.weights.h5")